# Spatial Transcriptomics Evaluation

Evaluate segmentation models on Xenium spatial transcriptomics data by measuring the **unassigned transcript fraction** (spatial bleeding) — the proportion of transcripts that fall outside mask boundaries.

**Datasets:**

| Dataset | License | Size |
|---|---|---|
| Human-pancreas | CC BY 4.0 | ~6.5 GB |
| Human-lung | CC BY 4.0 | ~19 GB |
| Mouse-colon | CC BY 4.0 | ~24 GB |

**Models:** microatlas, cellpose4, cellpose3, cellsam, microsam

## Data Download

Download the `*_outs` directory from [10x Genomics](https://www.10xgenomics.com/datasets) and place it under the corresponding dataset folder:

```
spatial_analysis/Xenium/
├── Human_pancreas/Xenium_V1_human_Pancreas_FFPE_outs/
├── Human_lung/Xenium_V1_humanLung_Cancer_FFPE_outs/
└── Mouse_colon/Xenium_V1_mouse_Colon_FF_outs/
```

In [ ]:
import sys
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd

# Setup paths
SRC_DIR = Path.cwd().parent / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
# Dataset and model selection
PROJECTS = ['Human_lung']  # Options: 'Human_lung', 'Human_pancreas', 'Mouse_colon'
MODEL = 'microatlas'       # Options: 'microatlas', 'cellpose4', 'cellpose3', 'cellsam', 'microsam'
USE_CROP = False           # Set True for quick testing on a small ROI
GPU_DEVICE = 0             # GPU index, -1 for CPU

XENIUM_BASE = SRC_DIR / 'spatial_analysis' / 'Xenium'

PROJECT_CONFIGS = {
    'Human_lung': {
        'data_dir': XENIUM_BASE / 'Human_lung' / 'Xenium_V1_humanLung_Cancer_FFPE_outs',
        'pixel_size': 0.2125,
    },
    'Human_pancreas': {
        'data_dir': XENIUM_BASE / 'Human_pancreas' / 'Xenium_V1_human_Pancreas_FFPE_outs',
        'pixel_size': 0.2125,
    },
    'Mouse_colon': {
        'data_dir': XENIUM_BASE / 'Mouse_colon' / 'Xenium_V1_mouse_Colon_FF_outs',
        'pixel_size': 0.2125,
    },
}

for proj in PROJECTS:
    cfg = PROJECT_CONFIGS[proj]
    exists = cfg['data_dir'].exists()
    print(f'  {proj}: {"OK" if exists else "NOT FOUND"} ({cfg["data_dir"]})')

## Run Segmentation Pipeline

For each dataset, run the segmentation pipeline which:
1. Loads the morphology_focus image
2. Runs the selected segmentation model
3. Saves mask (.npy) and segmentation overlay (.png)

**Output files per model:**

| Model | Mask file | Overlay |
|---|---|---|
| microatlas | `masks_microatlas_morphology.npy` | `seg_overlay_microatlas_morphology.png` |
| cellpose4 | `masks_cellpose4_morphology.npy` | `seg_overlay_cellpose4_morphology.png` |
| cellpose3 | `masks_cellpose3_morphology.npy` | `seg_overlay_cellpose3_morphology.png` |
| cellsam | `masks_cellsam_morphology.npy` | `seg_overlay_cellsam_morphology.png` |
| microsam | `masks_microsam_morphology.npy` | `seg_overlay_microsam_morphology.png` |

In [ ]:
for project in PROJECTS:
    print(f'\n{"="*60}')
    print(f'Project: {project}')
    print(f'Model: {MODEL}')
    print(f'{"="*60}')

    proj_cfg = PROJECT_CONFIGS[project]
    data_dir = proj_cfg['data_dir']
    pixel_size = proj_cfg['pixel_size']

    if not data_dir.exists():
        print(f'  [Skip] Data directory not found: {data_dir}')
        print(f'  Please download data from 10x Genomics first.')
        continue

    # Import project-specific modules
    proj_analysis_dir = XENIUM_BASE / project / 'microatlas_xenium_analysis'
    if str(proj_analysis_dir.parent) not in sys.path:
        sys.path.insert(0, str(proj_analysis_dir.parent))

    # Import config and modules for this project
    import importlib
    config_mod = importlib.import_module('microatlas_xenium_analysis.config')
    data_loader_mod = importlib.import_module('microatlas_xenium_analysis.data_loader')
    seg_mod = importlib.import_module('microatlas_xenium_analysis.segmentation')

    # Set output directory
    output_dir = proj_analysis_dir / 'output'
    output_dir.mkdir(exist_ok=True)
    config_mod.OUTPUT_DIR = output_dir
    config_mod.USE_CROP = USE_CROP
    config_mod.GPU_DEVICE = GPU_DEVICE

    # Load morphology image
    logger.info('Loading morphology image...')
    crop_roi = config_mod.CROP_ROI if USE_CROP else None
    morph_img = data_loader_mod.load_morphology_image(use_focus=True, crop_roi=crop_roi)
    logger.info(f'Morphology image shape: {morph_img.shape}')

    # Initialize segmentor
    if MODEL in ('cellpose4', 'microatlas'):
        if MODEL == 'microatlas':
            pretrained = str(SRC_DIR.parent / 'microatlas' / 'microatlas')
        else:
            pretrained = 'cpsam'
        segmentor = seg_mod.CPSAMSegmentor(
            gpu=(GPU_DEVICE >= 0),
            pretrained_model=pretrained,
            device=f'cuda:{GPU_DEVICE}' if GPU_DEVICE >= 0 else None,
        )
    elif MODEL == 'cellpose3':
        segmentor = seg_mod.Cellpose3Segmentor(
            gpu=(GPU_DEVICE >= 0),
            device=f'cuda:{GPU_DEVICE}' if GPU_DEVICE >= 0 else None,
        )
    elif MODEL == 'cellsam':
        segmentor = seg_mod.CellSAMSegmentor(
            gpu=(GPU_DEVICE >= 0),
            device=f'cuda:{GPU_DEVICE}' if GPU_DEVICE >= 0 else None,
        )
    elif MODEL == 'microsam':
        segmentor = seg_mod.MicroSAMSegmentor(
            gpu=(GPU_DEVICE >= 0),
            device=f'cuda:{GPU_DEVICE}' if GPU_DEVICE >= 0 else None,
        )

    # Run segmentation
    logger.info(f'Running {MODEL} segmentation...')
    t0 = time.time()
    mask, flows, styles = segmentor.segment_morphology(morph_img)
    elapsed = time.time() - t0
    logger.info(f'Segmentation completed in {elapsed:.1f}s')

    # Post-process: filter small masks
    mask = seg_mod.filter_masks(mask, min_size=15)
    logger.info(f'Found {mask.max()} cells')

    # Save mask
    mask_path = output_dir / f'masks_{MODEL}_morphology.npy'
    np.save(str(mask_path), mask)
    logger.info(f'Mask saved to {mask_path}')

## Compute Spatial Bleeding (Unassigned Transcript Fraction)

After all models finish segmentation, compute the unassigned transcript fraction for each model.
This measures the proportion of transcripts falling outside all segmentation mask pixels.

In [ ]:
BATCH_SIZE = 500_000

def compute_unassigned_fraction(mask, transcripts_df, pixel_size):
    """Compute fraction of transcripts outside all mask pixels."""
    x_px = np.round(transcripts_df['x_location'].values / pixel_size).astype(np.int32)
    y_px = np.round(transcripts_df['y_location'].values / pixel_size).astype(np.int32)

    h, w = mask.shape[:2]
    valid = (x_px >= 0) & (x_px < w) & (y_px >= 0) & (y_px < h)
    x_v, y_v = x_px[valid], y_px[valid]

    n_total = len(x_v)
    if n_total == 0:
        return 0.0, 0, 0

    inside = np.zeros(n_total, dtype=bool)
    for start in range(0, n_total, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_total)
        vals = mask[y_v[start:end], x_v[start:end]]
        inside[start:end] = vals > 0

    n_inside = inside.sum()
    n_outside = n_total - n_inside
    fraction = n_outside / n_total
    return fraction, n_inside, n_outside


# Evaluate spatial bleeding for each project
bleeding_results = []

for project in PROJECTS:
    proj_cfg = PROJECT_CONFIGS[project]
    data_dir = proj_cfg['data_dir']
    pixel_size = proj_cfg['pixel_size']

    if not data_dir.exists():
        continue

    # Load transcripts
    tx_path = data_dir / 'transcripts.parquet'
    if not tx_path.exists():
        print(f'  [Skip] Transcripts not found: {tx_path}')
        continue

    print(f'\n{project}:')
    tx_df = pd.read_parquet(tx_path, columns=['x_location', 'y_location'])
    print(f'  Total transcripts: {len(tx_df):,}')

    # Load mask for the model
    output_dir = XENIUM_BASE / project / 'microatlas_xenium_analysis' / 'output'
    mask_path = output_dir / f'masks_{MODEL}_morphology.npy'

    if mask_path.exists():
        mask = np.load(str(mask_path))
        fraction, n_in, n_out = compute_unassigned_fraction(mask, tx_df, pixel_size)
        print(f'  {MODEL}: unassigned={fraction:.4f} ({n_out:,} / {n_in + n_out:,})')
        bleeding_results.append({
            'Project': project, 'Model': MODEL,
            'Unassigned_fraction': fraction,
            'N_inside': n_in, 'N_outside': n_out,
        })
    else:
        print(f'  [Skip] Mask not found: {mask_path}')

if bleeding_results:
    df_bleeding = pd.DataFrame(bleeding_results)
    print(f'\nSpatial Bleeding Results:')
    print(df_bleeding.to_string(index=False))